# Appendix 3: Claude 3.5 Sonnet Implementation for Indicator Generation

This notebook presents the Appendix 3 workflow as an executable and auditable Jupyter notebook. It covers scenario summarization, initial indicator generation, indicator expansion, JSON validation, and output preservation.

## Documented implementation

- **Provider:** Amazon Bedrock
- **Region:** `eu-central-1`
- **Model identifier:** `anthropic.claude-3-5-sonnet-20240620-v1:0`
- **Primary tasks:**
  1. Summarize an input scenario.
  2. Generate initial red, blue, and predictive indicators.
  3. Expand each indicator using structured analytical attributes.
  4. Preserve machine-readable outputs for subsequent human review.

> **Reproducibility note:** AWS credentials are not embedded in the notebook. The operator must configure authorized credentials through the normal AWS credential chain. Exact dependency versions, source-document checksums, and any run-specific parameters should be recorded before execution.


## 1. Environment and dependencies

The original workflow uses Python with the AWS SDK for Python. This cell records the runtime environment and imports the libraries required for strict JSON processing and Bedrock invocation.


In [ ]:
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import boto3
from botocore.config import Config

print({
    "python": sys.version,
    "platform": platform.platform(),
    "boto3": boto3.__version__,
    "run_started_utc": datetime.now(timezone.utc).isoformat(),
})


## 2. Model and client configuration

The model and region below correspond to the Claude 3.5 Sonnet configuration documented in Appendix 3. Sampling parameters must be retained with each run because they affect reproducibility.


In [ ]:
AWS_REGION = "eu-central-1"
MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"

# Run parameters. Preserve these values in the run record.
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_TOKENS = 4096

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    config=Config(retries={"max_attempts": 3, "mode": "standard"}),
)


## 3. Input scenario

Paste the complete scenario or source text into `scenario_text`. For a reproducible run, retain a frozen copy of the source and calculate its SHA-256 checksum.


In [ ]:
scenario_text = r'''PASTE THE COMPLETE APPENDIX 3 INPUT SCENARIO HERE'''.strip()

if not scenario_text or scenario_text.startswith("PASTE THE COMPLETE"):
    print("Input scenario placeholder detected. Replace it before invoking the model.")


In [ ]:
import hashlib

scenario_sha256 = hashlib.sha256(scenario_text.encode("utf-8")).hexdigest()
print("Scenario SHA-256:", scenario_sha256)


## 4. Bedrock invocation helper

The helper sends a user prompt to Claude through the Bedrock Messages API and returns the generated text together with available usage metadata. Request and response material should be archived for audit and replication.


In [ ]:
def invoke_claude(prompt, *, max_tokens=MAX_TOKENS, temperature=TEMPERATURE, top_p=TOP_P):
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "messages": [
            {
                "role": "user",
                "content": [{"type": "text", "text": prompt}],
            }
        ],
    }

    response = bedrock.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(request_body),
        accept="application/json",
        contentType="application/json",
    )

    payload = json.loads(response["body"].read())
    text_parts = [
        block.get("text", "")
        for block in payload.get("content", [])
        if block.get("type") == "text"
    ]
    return {
        "text": "".join(text_parts).strip(),
        "usage": payload.get("usage", {}),
        "stop_reason": payload.get("stop_reason"),
        "model_id": MODEL_ID,
        "region": AWS_REGION,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


## 5. Scenario summarization

The first model call creates a concise analytical summary of the supplied scenario. The summary is an intermediate model output, not an independently verified factual product.


In [ ]:
summary_prompt = f'''You are supporting an indicators-and-warning analysis.

Read the scenario below and produce a concise, source-faithful summary. Preserve the principal actors, actions, conditions, chronology, objectives, and observable effects. Do not introduce facts that are absent from the scenario.

SCENARIO
{scenario_text}
'''

# Uncomment after supplying the complete scenario and configuring AWS credentials.
# summary_result = invoke_claude(summary_prompt)
# scenario_summary = summary_result["text"]
# print(scenario_summary)


## 6. Initial indicator generation

The second call generates three analytical classes:

- **Red indicators:** observable activity associated with a potential adversary, threat actor, or hostile campaign.
- **Blue indicators:** observable activity associated with friendly actors, institutions, systems, or populations.
- **Predictive indicators:** observable developments that may provide warning of a future event or transition.

The response is requested as strict JSON so that it can be validated and processed programmatically.


In [ ]:
indicator_prompt_template = '''You are an indicators-and-warning analyst.

Using only the supplied scenario summary and source scenario, generate candidate indicators in three categories:

1. red_indicators
2. blue_indicators
3. predictive_indicators

Return only valid JSON with this structure:
{{
  "red_indicators": [{{"name": "...", "description": "..."}}],
  "blue_indicators": [{{"name": "...", "description": "..."}}],
  "predictive_indicators": [{{"name": "...", "description": "..."}}]
}}

Requirements:
- Each indicator must describe an observable condition, action, event, or change.
- Keep indicators distinct and avoid duplicates.
- Do not add unsupported numerical probabilities.
- Do not add actors, mechanisms, or claims absent from the source.
- Treat the output as candidate analytical material requiring human review.

SCENARIO SUMMARY
{scenario_summary}

SOURCE SCENARIO
{scenario_text}
'''

# indicator_prompt = indicator_prompt_template.format(
#     scenario_summary=scenario_summary,
#     scenario_text=scenario_text,
# )
# indicator_result = invoke_claude(indicator_prompt)
# indicators_raw = indicator_result["text"]
# print(indicators_raw)


## 7. Strict JSON parsing and schema checks

Appendix 3's workflow should use `json.loads` rather than Python `eval`. Strict parsing is safer and makes the stated JSON requirement testable.


In [ ]:
REQUIRED_CATEGORIES = (
    "red_indicators",
    "blue_indicators",
    "predictive_indicators",
)


def parse_indicators(raw_text):
    parsed = json.loads(raw_text)
    if not isinstance(parsed, dict):
        raise ValueError("Indicator response must be a JSON object.")

    for category in REQUIRED_CATEGORIES:
        if category not in parsed:
            raise ValueError(f"Missing required category: {category}")
        if not isinstance(parsed[category], list):
            raise ValueError(f"{category} must be a list.")
        for index, item in enumerate(parsed[category]):
            if not isinstance(item, dict):
                raise ValueError(f"{category}[{index}] must be an object.")
            if not item.get("name") or not item.get("description"):
                raise ValueError(
                    f"{category}[{index}] requires non-empty name and description fields."
                )
    return parsed

# indicators = parse_indicators(indicators_raw)


## 8. Indicator expansion

Each candidate indicator is expanded into a structured analytical record. The attributes below reflect the Appendix 3 emphasis on observable evidence, collection, interpretation, and warning utility. Generated fields remain hypotheses until reviewed against the source material and relevant evidence.


In [ ]:
expansion_prompt_template = '''You are an indicators-and-warning analyst.

Expand the candidate indicator below as a structured analytical record. Return only valid JSON.

CATEGORY
{category}

INDICATOR
Name: {name}
Description: {description}

SOURCE SCENARIO
{scenario_text}

Return this structure:
{{
  "name": "...",
  "category": "...",
  "description": "...",
  "observable_evidence": ["..."],
  "collection_sources": ["..."],
  "collection_method": ["..."],
  "frequency_or_timing": "...",
  "threshold_or_change_condition": "...",
  "analytical_significance": "...",
  "alternative_explanations": ["..."],
  "limitations": ["..."],
  "source_support": "direct, inferred, or unsupported",
  "human_review_required": true
}}

Requirements:
- Remain faithful to the source scenario.
- Clearly distinguish direct source support from inference.
- Do not fabricate probabilities, named entities, technical mechanisms, or evidence.
- Describe feasible collection sources without implying that collection was performed.
'''


In [ ]:
def expand_indicator(category, indicator, scenario_text):
    prompt = expansion_prompt_template.format(
        category=category,
        name=indicator["name"],
        description=indicator["description"],
        scenario_text=scenario_text,
    )
    result = invoke_claude(prompt)
    expanded = json.loads(result["text"])
    return {"record": expanded, "run_metadata": result}


def expand_all_indicators(indicators, scenario_text):
    expanded_records = []
    for category in REQUIRED_CATEGORIES:
        for indicator in indicators[category]:
            expanded_records.append(
                expand_indicator(category, indicator, scenario_text)
            )
    return expanded_records

# expanded_indicators = expand_all_indicators(indicators, scenario_text)


## 9. Human review record

The model output should not be described as validated merely because it parsed successfully. A human author or analyst should record whether each item is supported, needs revision, or should be rejected.


In [ ]:
def make_review_record(expanded_item):
    record = expanded_item["record"]
    return {
        "indicator_name": record.get("name"),
        "reviewer_role": "",
        "source_fidelity": "not_reviewed",
        "predictive_value": "not_reviewed",
        "diagnostic_value": "not_reviewed",
        "unambiguity": "not_reviewed",
        "collectability": "not_reviewed",
        "unsupported_specificity": "not_reviewed",
        "disposition": "not_reviewed",  # accepted, revised, or rejected
        "review_notes": "",
        "reviewed_utc": None,
    }

# human_review_records = [make_review_record(x) for x in expanded_indicators]


## 10. Save the reproducibility package

This cell saves prompts, model metadata, source checksum, raw outputs, parsed records, and review records. Credentials and secrets must never be written to the package.


In [ ]:
def save_json(path, value):
    path = Path(path)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return path


def build_run_record(summary_result, indicator_result, indicators, expanded_indicators, reviews):
    return {
        "run_id": datetime.now(timezone.utc).strftime("appendix3-%Y%m%dT%H%M%SZ"),
        "model_id": MODEL_ID,
        "provider": "Amazon Bedrock",
        "region": AWS_REGION,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_TOKENS,
        "scenario_sha256": scenario_sha256,
        "summary_prompt": summary_prompt,
        "summary_result": summary_result,
        "indicator_prompt": indicator_prompt,
        "indicator_result": indicator_result,
        "parsed_indicators": indicators,
        "expanded_indicators": expanded_indicators,
        "human_review_records": reviews,
    }

# run_record = build_run_record(
#     summary_result,
#     indicator_result,
#     indicators,
#     expanded_indicators,
#     human_review_records,
# )
# output_path = save_json("appendix3_run_record.json", run_record)
# print("Saved:", output_path.resolve())


## 11. End-to-end execution cell

After supplying the complete scenario and confirming the model configuration, this cell executes the full workflow. It remains commented to prevent accidental API use.


In [ ]:
# summary_result = invoke_claude(summary_prompt)
# scenario_summary = summary_result["text"]
#
# indicator_prompt = indicator_prompt_template.format(
#     scenario_summary=scenario_summary,
#     scenario_text=scenario_text,
# )
# indicator_result = invoke_claude(indicator_prompt)
# indicators_raw = indicator_result["text"]
# indicators = parse_indicators(indicators_raw)
#
# expanded_indicators = expand_all_indicators(indicators, scenario_text)
# human_review_records = [make_review_record(x) for x in expanded_indicators]
#
# run_record = build_run_record(
#     summary_result,
#     indicator_result,
#     indicators,
#     expanded_indicators,
#     human_review_records,
# )
# save_json("appendix3_run_record.json", run_record)


## 12. Replication checklist

Before treating a run as reproducible, record the following:

- Frozen source document and SHA-256 checksum
- Repository URL, branch, and Git commit SHA
- Local code modifications
- Python and dependency versions
- Exact model identifier and provider region
- All inference parameters and provider defaults
- Full prompts and substitutions
- Raw responses, retries, errors, and token usage
- Parsing and post-processing procedures
- Human reviewers, criteria, changes, and final dispositions

## Interpretation boundary

This workflow demonstrates a method for producing candidate indicators from unstructured text. It does not, by itself, establish operational accuracy, predictive validity, or empirical verification. Those conclusions require independent human annotation, repeated trials, and evaluation against a suitable reference set.
